## 1. Intro

### Agent Framework

This notebook shows how to author a LangGraph agent that connects to an AI Search Index hosted on Databricks. LangGraph's graph-based architecture gives you complete control over agent behavior, making it the right choice when you need custom workflows or multi-step reasoning patterns.

In this notebook, you:

- Author a LangGraph agent
- [Added] Connect to the vector search index to perform RAG.
- Test the agent and evaluate its responses using MLflow Evaluation
- Log the agent with MLflow and deploy it to a model serving endpoint

This notebook uses the  [`ResponsesAgent`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pyfunc.html#mlflow.pyfunc.ResponsesAgent) for Databrick compatibility.

To learn more about authoring an agent using Agent Framework, see Databricks documentation ([AWS](https://docs.databricks.com/aws/generative-ai/agent-framework/author-agent) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/create-chat-model)).

### Prerequisites

- Address all `TODO`s in this notebook.
- Added `langgraph-prebuilt==1.0.8` dependency as the temp workaround for the issue discussed here: https://github.com/langchain-ai/langgraph/issues/7404

### Source
https://docs.databricks.com/aws/en/notebooks/source/generative-ai/langgraph-mcp-tool-calling-agent.html

## 2. Define the agent code

Define the agent code in a single cell below. This lets you easily write the agent
code to a local Python file, using the `%%writefile` magic command, for subsequent
logging and deployment.

**What this code does at a high level:**

1. **Connect to MCP servers using adapters**
    The `DatabricksMCPServer` and `DatabricksMultiServerMCPClient` from `databricks_langchain` handle:
    - Connections to Databricks MCP servers
    - Authentication
    - Automatic tool discovery and conversion to LangChain-compatible format

2. **Build a LangGraph agent workflow using LangGraph `StateGraph`**

5. **Wrap with ResponsesAgent**
    The agent is wrapped using `ResponsesAgent` for compatibility with Databricks
    features like evaluation, deployment, and feedback collection.

6. **MLflow autotracing**
    Enable MLflow autologging to automatically trace LLM calls, tool invocations,
    and agent state transitions.

#### Agent tools

This example connects to:
- Vector search MCP servers (for semantic search over your data)
- [added] Vector search index

In [0]:
%pip install -U -qqqq --force-reinstall databricks-langchain databricks-agents uv "langgraph-prebuilt==1.0.8" 

In [0]:
dbutils.library.restartPython()

In [0]:
%%writefile agent.py

import asyncio
from typing import Annotated, Any, AsyncGenerator, Generator, Optional, Sequence, TypedDict, Union

import mlflow
import nest_asyncio
from databricks.sdk import WorkspaceClient
from databricks_langchain import (
    ChatDatabricks,
    DatabricksMCPServer,
    DatabricksMultiServerMCPClient,
    VectorSearchRetrieverTool, # added
)
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from langchain_core.messages.tool import ToolMessage
import json

nest_asyncio.apply()


LLM_ENDPOINT_NAME = "databricks-qwen35-122b-a10b"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

# TODO: Update with your system prompt
system_prompt = """
    You are the knowledge assistant for Singapore's Infocomm Media Development Authority (IMDA), specialized on  testing LLM-based applications for safety and reliability. Respond in a clear, professional, and factual tone appropriate for developers. Use only verified information from the internal documents, and include source references when available. If the answer cannot be found, clearly state that, and suggest related sections or next steps. Do not speculate, make assumptions, or provide informaiton outside of the provided context.
"""

workspace_client = WorkspaceClient()

host = workspace_client.config.host
databricks_mcp_client = DatabricksMultiServerMCPClient(
    [
        DatabricksMCPServer(
            name="system-ai",
            url=f"{host}/api/2.0/mcp/functions/system/ai",
        ),
    ]
)

catalog = "workspace"
schema = "feature_model"
index = "docs_chunked_index"
INDEX_NAME = f"{catalog}.{schema}.{index}"
NUM_RESULTS = 3

databricks_vector_search = VectorSearchRetrieverTool(
    name="imda_llm_testing_knowledge_search",
    index_name=INDEX_NAME,
    description="Search the IMDA's document `Starter Kit for Testing LLM-Based Applications for Safety and Reliability` for relevant information on testing LLM-based applications.",
    num_results=NUM_RESULTS
)


# The state for the agent workflow, including the conversation and any custom data
class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    system_prompt: Optional[str] = None,
):
    model = model.bind_tools(tools)  # Bind tools to the model

    # A. NODE DEFINITIONS:
    
    # A1. The control flow
    # Function to check if agent should continue or finish based on last message
    def should_continue(state: AgentState):
        messages = state["messages"]
        last_message = messages[-1]
        # If function (tool) calls are present, continue; otherwise, end
        if isinstance(last_message, AIMessage) and last_message.tool_calls:
            return "continue"
        else:
            return "end"

    # A2. The main agent node:
    # Preprocess: optionally prepend a system prompt to the conversation history
    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])

    model_runnable = preprocessor | model  # Chain the preprocessor and the model

    # The function to invoke the model within the workflow
    def call_model(
        state: AgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    # B. GRAPH DEFINITION:
    # create the agent with ReAct pattern:
    workflow = StateGraph(AgentState)  # Create the agent's state machine
    workflow.add_node("agent", RunnableLambda(call_model))  # Agent node (LLM)
    workflow.add_node("tools", ToolNode(tools))  # Tools node
    workflow.set_entry_point("agent")  # Start at agent node
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",  # If the model requests a tool call, move to tools node
            "end": END,  # Otherwise, end the workflow
        },
    )
    workflow.add_edge("tools", "agent")  # After tools are called, return to agent node

    # Compile and return the tool-calling agent workflow
    return workflow.compile()


# ResponsesAgent class to wrap the compiled agent and make it compatible with Databricks Responses API
class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, agent):
        self.agent = agent

    # Make a prediction (single-step) for the agent
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done" or event.type == "error"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    async def _predict_stream_async(
        self,
        request: ResponsesAgentRequest,
    ) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])
        # Stream events from the agent graph
        async for event in self.agent.astream(
            {"messages": cc_msgs}, stream_mode=["updates", "messages"]
        ):
            if event[0] == "updates":
                # Stream updated messages from the workflow nodes
                for node_data in event[1].values():
                    if len(node_data.get("messages", [])) > 0:
                        all_messages = []
                        for msg in node_data["messages"]:
                            if isinstance(msg, ToolMessage) and not isinstance(msg.content, str):
                                msg.content = json.dumps(msg.content)
                            all_messages.append(msg)
                        for item in output_to_responses_items_stream(all_messages):
                            yield item
            elif event[0] == "messages":
                # Stream generated text message chunks
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=content, item_id=chunk.id),
                        )
                except:
                    pass

    # Stream predictions for the agent, yielding output as it's generated
    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        agen = self._predict_stream_async(request)

        try:
            loop = asyncio.get_event_loop()
        except RuntimeError:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)

        ait = agen.__aiter__()

        while True:
            try:
                item = loop.run_until_complete(ait.__anext__())
            except StopAsyncIteration:
                break
            else:
                yield item


# Initialize the entire agent, including MCP tools and workflow
def initialize_agent():
    """Initialize the agent with MCP tools"""
    tools = []

    # Create MCP tools from the configured servers
    mcp_tools = asyncio.run(databricks_mcp_client.get_tools())
    tools.extend(mcp_tools)

    tools.append(databricks_vector_search)

    # Create the agent graph with an LLM, tool set, and system prompt (if given)
    agent = create_tool_calling_agent(llm, tools, system_prompt)
    return LangGraphResponsesAgent(agent)


mlflow.langchain.autolog()
AGENT = initialize_agent()
mlflow.models.set_model(AGENT)

## 3. Set the Experiment for in-Prod Tracing

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow

# set a single experiment for all related activities
experiment = "prd_imda_langgraph_knowledge_assistant"
experiment_name = f"/Workspace/Shared/{experiment}"
mlflow.set_experiment(experiment_name)

## 4. Test the agent

Interact with the agent to test its output and tool-calling abilities. Since this notebook called `mlflow.langchain.autolog()`, you can view the trace for each step the agent takes.

In [0]:
from agent import AGENT

result = AGENT.predict({"input": [{"role": "user", "content": "What is 15*6 in Python"}], "custom_inputs": {"session_id": "test-session"}})
print(result.model_dump(exclude_none=True))

In [0]:
result = AGENT.predict({"input": [{"role": "user", "content": "What is the purpose of this IMDA LLM testing starter kit?"}], "custom_inputs": {"session_id": "test-session"}})
print(result.model_dump(exclude_none=True))

In [0]:
for chunk in AGENT.predict_stream(
    {"input": [{"role": "user", "content": "what is RabakBench"}], "custom_inputs": {"session_id": "test-session-stream"}}
):
    print(chunk, "-----------\n")

In [0]:
from IPython.display import Image, display
from agent import AGENT

print(type(AGENT.agent)) # get the underlying CompiledStateGraph object from the ResponsesAgent subclass
display(Image(AGENT.agent.get_graph().draw_mermaid_png()))

## 5. Log the agent as an MLflow model

Log the agent as code from the `agent.py` file. See [Deploy an agent that connects to Databricks MCP servers](https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp#deploy-your-agent).

In [0]:
from agent import databricks_vector_search

databricks_vector_search.resources

In [0]:
import mlflow
from agent import LLM_ENDPOINT_NAME
from agent import databricks_vector_search
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction
from pkg_resources import get_distribution

resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),
    DatabricksFunction(function_name="system.ai.python_exec")
]
# to avoid the need of init the vector search client again, we can just use the existing one defined 
# inside the agent code
resources.extend(databricks_vector_search.resources)

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        resources=resources,
        pip_requirements=[
            f"langgraph=={get_distribution('langgraph').version}",
            f"mcp=={get_distribution('mcp').version}",
            f"databricks-mcp=={get_distribution('databricks-mcp').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph-prebuilt=={get_distribution('langgraph-prebuilt').version}",
        ]
    )

## 6. Evaluate the agent with [Agent Evaluation](https://docs.databricks.com/mlflow3/genai/eval-monitor)

You can edit the requests or expected responses in your evaluation dataset and run evaluation as you iterate your agent, leveraging mlflow to track the computed quality metrics.

Evaluate your agent with one of our [predefined LLM scorers](https://docs.databricks.com/mlflow3/genai/eval-monitor/predefined-judge-scorers), or try adding [custom metrics](https://docs.databricks.com/mlflow3/genai/eval-monitor/custom-scorers).

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, Safety, RetrievalRelevance, RetrievalGroundedness

eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "Calculate the 15th Fibonacci number"}]},
        "expected_response": "The 15th Fibonacci number is 610.",
    },
    {
        "inputs": {"input": [{"role": "user", "content": "what is the purpose of this IMDA LLM testing starter kit?"}]},
        "expected_response": "The Starter Kit for Testing LLM-Based Applications for Safety and Reliability, developed by Singapore's Infocomm Media Development Authority (IMDA) in collaboration with the AI Verify Foundation, serves as a set of voluntary guidelines. By codifying these standards, the Starter Kit aims to contribute to the growth of a trusted, secure, and reliable AI ecosystem.",
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input, "custom_inputs": {"session_id": "evaluation-session"}}),
    scorers=[RelevanceToQuery(), Safety()], # add more scorers here if they're applicable
)

# Review the evaluation results in the MLfLow UI (see console output)

## 7. Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "Hello!"}], "custom_inputs": {"session_id": "validation-session"}},
    env_manager="uv",
)

## 7. Register the model to Unity Catalog

Before you deploy the agent, you must register the agent to Unity Catalog.

- **TODO** Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "workspace"
schema = "feature_model"
model_name = "imda_langgraph_knowledge_assistant"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

## 8. Deploy the agent

NOTE: for Databricks Free Edition, from observation, ensure after this deployment:
- there are maximum 2 endpoints present in the Serving endpoint list, regardless of status. Else the deployment will timeout after 5min due to limit on free edition.
- Max 3x served entities across all endpoints, each having 0-4 provisioned concurrency

In [0]:
from databricks import agents
import mlflow


current_experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
print(current_experiment_id)
environment_vars = {
    "MLFLOW_EXPERIMENT_ID": current_experiment_id,
}


agents.deploy(
    model_name = UC_MODEL_NAME,
    model_version=uc_registered_model_info.version,
    endpoint_name=model_name,
    # ==============================================================================
    # TODO: ONLY UNCOMMENT AND CONFIGURE THE ENVIRONMENT_VARS SECTION BELOW
    #       IF YOU ARE USING OAUTH/SERVICE PRINCIPAL FOR CUSTOM MCP SERVERS.
    #       For managed MCP (the default), LEAVE THIS SECTION COMMENTED OUT.
    # ==============================================================================
    # environment_vars={
    #     "DATABRICKS_CLIENT_ID": DATABRICKS_CLIENT_ID,
    #     "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{client_secret_scope_name}/{client_secret_key_name}}}}}"
    # },
    deploy_feedback_model=False,
    scale_to_zero=True,
    environment_vars=environment_vars,
)
